# ️ Glu-Stock: 03_EXECUTION_MONITOR
**Phase**: Command & Control | v19.1 (Institutional Sync)

This notebook reports trade plans generated dynamically by the LLM Brain.

In [ ]:
!pip install -q requests firebase-admin yfinance pandas ta matplotlib


In [ ]:
import requests, json, os, firebase_admin, pandas as pd, time, matplotlib.pyplot as plt
from firebase_admin import credentials, firestore
from datetime import datetime, timedelta, timezone
from kaggle_secrets import UserSecretsClient
import yfinance as yf


In [ ]:
def send_telegram(msg, photo_path=None):
    secrets = UserSecretsClient()
    token = secrets.get_secret("TELEGRAM_TOKEN")
    chat_id = secrets.get_secret("TELEGRAM_CHAT_ID")
    
    if photo_path and os.path.exists(photo_path):
        url = f"https://api.telegram.org/bot{token}/sendPhoto"
        try:
            with open(photo_path, 'rb') as photo:
                requests.post(url, data={"chat_id": chat_id, "caption": msg, "parse_mode": "Markdown"}, files={"photo": photo})
        except: pass
    else:
        url = f"https://api.telegram.org/bot{token}/sendMessage"
        requests.post(url, json={"chat_id": chat_id, "text": msg, "parse_mode": "Markdown"})

def generate_strategy_plot(ticker, df, plan):
    plt.figure(figsize=(10, 6))
    plt.plot(df.index, df['Close'], label='Price', color='black', alpha=0.7)
    plt.axhline(y=plan['buy_price'], color='blue', linestyle='--', label=f'Buy: {plan["buy_price"]}')
    plt.axhline(y=plan['tp_price'], color='green', linestyle='--', label=f'TP: {plan["tp_price"]}')
    plt.axhline(y=plan['sl_price'], color='red', linestyle='--', label=f'SL: {plan["sl_price"]}')
    plt.title(f"Glu-Stock Visualizer: {ticker}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    path = f"{ticker}_plan.png"
    plt.savefig(path)
    plt.close()
    return path

def wait_until_0830_wib():
    print("Waiting until 08:30 WIB for report execution...", flush=True)
    while True:
        now_wib = datetime.now(timezone.utc) + timedelta(hours=7)
        target_wib = now_wib.replace(hour=8, minute=30, second=0, microsecond=0)
        if now_wib >= target_wib: 
            print(f"Target reached: {now_wib}")
            return
        time.sleep(60)

def generate_legendary_report(db):
    ihsg = yf.download('^JKSE', period='5d', progress=False)
    report_header = f"🏆 *GLU-STOCK LEGENDARY RECAP (v19.1)*\n📅 {datetime.now().strftime('%d %b %Y | 08:30 WIB')}\n🌍 IHSG: {ihsg['Close'].iloc[-1]:,.2f}\n\n"
    
    finals = list(db.collection("glu_stock_final_signals").get())
    if not finals:
        send_telegram(report_header + "- No high-conviction plans generated today.")
    else:
        for doc in finals:
            d = doc.to_dict()
            plan = d.get('plan', {})
            metrics = plan.get('metrics', {})
            
            msg = f"✅ *{d.get('ticker')} STRATEGY PLAN*\n"
            msg += f"   - Buy: {plan.get('buy_price')}\n"
            msg += f"   - TP : {plan.get('tp_price')}\n"
            msg += f"   - SL : {plan.get('sl_price')}\n\n"
            msg += f"📊 *Metrics*: RSI={metrics.get('rsi')}, ATR={metrics.get('atr')}\n"
            msg += f"🧠 *Reasoning*: {plan.get('reasoning')[:250]}..."
            
            # Generate visual
            df = yf.download(d.get('ticker'), period='30d', progress=False)
            photo_p = generate_strategy_plot(d.get('ticker'), df, plan)
            
            send_telegram(msg, photo_path=photo_p)
            doc.reference.delete()
    
    print("Execution Recap Sent.")

def run_execution_monitor():
    secrets = UserSecretsClient()
    cred_json = json.loads(secrets.get_secret("FIREBASE_KEY_JSON"))
    if not firebase_admin._apps: firebase_admin.initialize_app(credentials.Certificate(cred_json))
    db = firestore.client()
    
    print("Waiting for Strategic Planning (02)...", flush=True)
    wait_start = time.time()
    while time.time() - wait_start < 7200: # Wait up to 2 hours
        state = db.collection("glu_stock_state").document("last_run").get()
        if state.exists and state.to_dict().get("stage") == "02": break
        time.sleep(60)
    
    wait_until_0830_wib()
    generate_legendary_report(db)

if __name__ == "__main__":
    run_execution_monitor()
